In [42]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [43]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langgraph.graph import StateGraph, START, END
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.tools import tool
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import tool_node, tools_condition, ToolNode

In [44]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile"
)

llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002388C1F6C90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002388C1F6250>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [45]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9363.30it/s]


In [46]:
loader = PyPDFLoader("AI_ML.pdf")
docs = loader.load()

In [47]:
len(docs)

22

In [48]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = splitter.split_documents(docs)

In [49]:
len(chunks)

42

In [50]:
vector_store = Chroma.from_documents(chunks, embeddings)

In [51]:
vector_store

In [52]:
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

In [53]:
@tool
def rag_tool(query):
    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual / conceptual questions
    that might be answered from the stored documents.    
    """
    
    result = retriever.invoke(query)
    
    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]
    
    return {
        'query':query,
        'context':context,
        'metadata': metadata
    }

In [54]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [55]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [56]:
def chat_node(state: ChatState):
    messages = state['messages']
    
    response = llm_with_tools.invoke(messages)
    
    return {'messages': [response]}
    

In [57]:
tool_node = ToolNode(tools)

In [58]:
graph = StateGraph(ChatState)

In [59]:
graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

In [60]:
graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')


chatbot = graph.compile()

In [64]:
result = chatbot.invoke(
    {
    "messages": [
        HumanMessage(
            content=("Using the pdf notes, Explain how aliens enter in the earth" )
        )
    ]
})

In [65]:
print(result['messages'][-1].content)

There is no information in the pdf notes about aliens entering Earth. The provided text appears to be related to artificial intelligence and machine learning, and does not contain any relevant information about aliens or their potential methods of entering Earth.
